# **Comparing performances among different file formats**

**Setup: Install pyarrow if not present (usually pre-installed in Colab)**

In [1]:

!pip install -q pyarrow fastparquet

import pandas as pd
import numpy as np
import time
import os
import shutil

print("Libraries imported successfully.")

Libraries imported successfully.


**Dataset preparation**

In [2]:

np.random.seed(42)

n_rows = 5_000_000

print(f"Generating {n_rows} rows. Please wait...")

data = {
    'transaction_id': np.arange(n_rows),
    'timestamp': pd.date_range(start='2024-01-01',end='2026-03-31', periods=n_rows),
    'user_id': np.random.randint(10000, 99999, size=n_rows),
    'product_category': np.random.choice(['Electronics', 'Home', 'Books', 'Clothing', 'Garden'], n_rows),
    'price': np.random.uniform(10.0, 1000.0, size=n_rows)

}

df = pd.DataFrame(data)
# Add a 'date' column for the partitioning exercise later


print("Dataset ready.")
df.head()

Generating 5000000 rows. Please wait...
Dataset ready.


,transaction_id,timestamp,user_id,product_category,price
0,0,2024-01-01 00:00:00.000000000,25795,Electronics,439.592591
1,1,2024-01-01 00:00:14.169602833,10860,Books,520.298979
2,2,2024-01-01 00:00:28.339205667,86820,Books,68.384913
3,3,2024-01-01 00:00:42.508808501,64886,Books,651.354832
4,4,2024-01-01 00:00:56.678411335,16265,Garden,662.041412


 **Saving in different formats and comparing the sizes**

In [3]:
# Save as CSV
df.to_csv('data.csv', index=False)

# Save as Parquet
df.to_parquet('data.parquet', engine='pyarrow', compression='snappy')

# Get file sizes in MB
csv_size = os.path.getsize('data.csv') / (1024**2)
pq_size = os.path.getsize('data.parquet') / (1024**2)

print(f"CSV Size: {csv_size:.2f} MB")
print(f"Parquet Size: {pq_size:.2f} MB")
print(f"Space Saved: {100 - (pq_size/csv_size*100):.2f}%")

CSV Size: 332.54 MB
Parquet Size: 113.13 MB
Space Saved: 65.98%


# **Comparing csv and parquet in reading**

 CSV requires reading the whole file to find the columns, Parquet allows reading ONLY the necessary columns: 'product_category' and 'price'

In [4]:

start_time = time.time()

csv_df = pd.read_csv('data.csv')
electronics_revenue_csv = csv_df[csv_df['product_category'] == 'Electronics']['price'].sum()
csv_duration = time.time() - start_time
print(csv_duration)


start_time = time.time()

import pyarrow.parquet as pq
table = pq.read_table('data.parquet', columns=['product_category', 'price'])
pq_df = table.to_pandas()
electronics_revenue_pq = pq_df[pq_df['product_category'] == 'Electronics']['price'].sum()
pq_duration = time.time() - start_time

print(f"CSV Query Time: {csv_duration:.4f} seconds")
print(f"Parquet Query Time: {pq_duration:.4f} seconds")
print(f"Speedup Factor: {csv_duration / pq_duration:.2f}x")

3.970806360244751
CSV Query Time: 3.9708 seconds
Parquet Query Time: 0.5431 seconds
Speedup Factor: 7.31x


 Partioning the parquet file

In [5]:

if os.path.exists('data_partitioned'):
    shutil.rmtree('data_partitioned')

df.to_parquet('data_partitioned', partition_cols=['product_category'], engine='pyarrow')

#  list the directory to see the structure
print("Partitioned Directory Structure:")
!ls -R data_partitioned

#  Let's query a  specific category
start_time = time.time()

partitioned_df = pd.read_parquet('data_partitioned/product_category=Garden')
garden_revenue = partitioned_df['price'].sum()
partitioned_duration = time.time() - start_time

garden_revenue_pq = pq_df[pq_df['product_category'] == 'Garden']['price'].sum()
pq_duration = time.time() - start_time

print(f"\nPartitioned Query Time (Garden only): {partitioned_duration:.4f} seconds")
print(f"Parquet Query Time (Garden Category): {pq_duration:.4f} seconds")
print(f"Speedup Factor: { pq_duration/partitioned_duration:.2f}x")

Partitioned Directory Structure:
data_partitioned:
'product_category=Books'	'product_category=Garden'
'product_category=Clothing'	'product_category=Home'
'product_category=Electronics'

'data_partitioned/product_category=Books':
a23280a2f2844dd3b4d6df80e6c3932a-0.parquet

'data_partitioned/product_category=Clothing':
a23280a2f2844dd3b4d6df80e6c3932a-0.parquet

'data_partitioned/product_category=Electronics':
a23280a2f2844dd3b4d6df80e6c3932a-0.parquet

'data_partitioned/product_category=Garden':
a23280a2f2844dd3b4d6df80e6c3932a-0.parquet

'data_partitioned/product_category=Home':
a23280a2f2844dd3b4d6df80e6c3932a-0.parquet

Partitioned Query Time (Garden only): 0.0789 seconds
Parquet Query Time (Garden Category): 0.4151 seconds
Speedup Factor: 5.26x


We gained in time, but what abouth the partitioned files cumulative size?

In [ ]:
# Function to calculate directory size
def get_dir_size(path='.'):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size


full_pq_size = os.path.getsize('data.parquet') / (1024**2)
partitioned_pq_size = get_dir_size('data_partitioned') / (1024**2)

print(f"Single Parquet File Size: {full_pq_size:.2f} MB")
print(f"Partitioned Directory Total Size: {partitioned_pq_size:.2f} MB")
print(f"Size Difference: {partitioned_pq_size - full_pq_size:.2f} MB ({(partitioned_pq_size/full_pq_size - 1)*100:.2f}% change)")


print("\n--- Breakdown by Partition ---")
partitions = [d for d in os.listdir('data_partitioned') if os.path.isdir(os.path.join('data_partitioned', d))]
for p in sorted(partitions):
    p_size = get_dir_size(os.path.join('data_partitioned', p)) / (1024**2)
    print(f"Partition '{p}': {p_size:.2f} MB")

Single Parquet File Size: 113.25 MB
Partitioned Directory Total Size: 147.82 MB
Size Difference: 34.56 MB (30.52% change)

--- Breakdown by Partition ---
Partition 'product_category=Books': 29.56 MB
Partition 'product_category=Clothing': 29.57 MB
Partition 'product_category=Electronics': 29.54 MB
Partition 'product_category=Garden': 29.57 MB
Partition 'product_category=Home': 29.57 MB


**Comment 1**

Notice that the partitioned data might be  larger than the single file. This is obvious: each small file in the partitions must store its own metadata and schema headers. In Data Infrastructure, this is the "Small File Problem" if partitions become too granular.

**Comment 2**

The Partitioned Query is  faster because of Partition Pruning. The storage engine doesn't  "open" the  other folders and it goes straight to the Garden one.  Big Data engines like Hive, Presto, and Spark handle Petabytes in this way.